## Importing

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
import math
import requests
from torch.utils.data import Dataset, DataLoader

## Loading Data Set

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
chars_len = len(chars)

encoded_text = [char_to_int[ch] for ch in text]

print(f'Text length: {len(text)}')
print(f'No. of Characters: {chars_len}')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Text length: 1115394
No. of Characters: 65
Using device: cuda


## Creating the Dataset Class

In [3]:
class CharDataSet(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, index):
        return self.sequences[index], self.targets[index]

def build_loaders(encoded_text, seq_len, batch_size=128):
    sequences = []
    targets = []

    for i in range(len(encoded_text) - seq_len):
        sequences.append(encoded_text[i: i+seq_len])
        targets.append(encoded_text[i + seq_len])

    sequences = torch.tensor(sequences, dtype=torch.long)
    targets = torch.tensor(targets, dtype=torch.long)

    dataset = CharDataSet(sequences, targets)
    train_size = int(len(dataset) * 0.8)
    test_size = len(dataset) - train_size

    train_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, shuffle=True, batch_size=batch_size)
    test_loader = DataLoader(test_ds, shuffle=False, batch_size=batch_size)

    return train_loader, test_loader 

## LSTM Model

In [4]:
class modelLSTM(nn.Module):
    def __init__(self, embed_size, hid_size, num_layers, fc_hidden=None):
        super().__init__()

        self.embedding = nn.Embedding(chars_len, embed_size)
        self.lstm = nn.LSTM(embed_size, hid_size, num_layers=num_layers, batch_first=True)

        if fc_hidden:
            self.fc = nn.Sequential(nn.Linear(hid_size, fc_hidden), nn.ReLU(), nn.Linear(fc_hidden, chars_len))
        else:
            self.fc = nn.Linear(hid_size, chars_len)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.fc(output[:, -1, :])
        
        return output

## GRU Model

In [5]:
class modelGRU(nn.Module):
    def __init__(self, embed_size, hid_size, num_layers, fc_hidden=None):
        super().__init__()

        self.embedding = nn.Embedding(chars_len, embed_size)
        self.gru = nn.GRU(embed_size, hid_size, num_layers=num_layers, batch_first=True)

        if fc_hidden:
            self.fc = nn.Sequential(nn.Linear(hid_size, fc_hidden), nn.ReLU(), nn.Linear(fc_hidden, chars_len))
        else:
            self.fc = nn.Linear(hid_size, chars_len)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.gru(embedded)
        output = self.fc(output[:, -1, :])
        
        return output

# Helper Functions

In [6]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [7]:
def model_size(model):
    total = sum(p.nelement() * p.element_size() for p in model.parameters())
    total += sum(b.nelement() * b.element_size() for b in model.buffers())

    model_size = total / 1024 / 1024

    return model_size

In [8]:
def gen_text(model, seed_str, seq_len, n_chars=100):
    model.eval()
    result = seed_str

    with torch.no_grad():
        for _ in range(n_chars):
            context = result[-seq_len:].ljust(seq_len)
            x   = torch.tensor([[char_to_int[c] for c in context]], dtype=torch.long).to(device)
            out = model(x)
            idx = int(torch.argmax(out, dim=1).item())
            result += int_to_char[idx]

    return result

In [9]:
def measure_inference_time(model, test_loader, n_runs=10):
    model.eval()
    batch = next(iter(test_loader))[0][:32].to(device)

    with torch.no_grad():
        for _ in range(3):
            model(batch)
    
    t0 = time.time()
    
    with torch.no_grad():
        for _ in range(n_runs):
            model(batch)
    
    return (time.time() - t0) / n_runs

In [10]:
def train_model(model, train_loader, test_loader, epochs=30, lr=0.005):
    criteria = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    train_losses, val_losses, val_accuracies = [], [], []
 
    t0 = time.time()

    for epoch in range(epochs):
 
        # training loop
        model.train()
        epoch_loss, n_batches = 0.0, 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            train_loss = criteria(model(X_batch), y_batch)
            train_loss.backward()
            optimizer.step()
            epoch_loss += train_loss.item()
            n_batches  += 1

        avg_train_loss = epoch_loss / n_batches
        train_losses.append(avg_train_loss)
 
        # Validation Loop
        model.eval()
        correct, total, val_loss_sum, val_batches = 0, 0, 0.0, 0

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
    
                out      = model(X_batch)
                val_loss_sum += criteria(out, y_batch).item()
                _, pred  = torch.max(out, 1)
                correct += (pred == y_batch).sum().item()
                total   += y_batch.size(0)
                val_batches += 1

        avg_val_loss = val_loss_sum / val_batches
        acc = correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(acc)
 
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d} | train loss {avg_train_loss:.4f} | "
                  f"val loss {avg_val_loss:.4f} | val acc {acc:.4f}")
 
    train_time = time.time() - t0
    return {
        "train_losses"    : train_losses,
        "val_losses"      : val_losses,
        "val_accs"        : val_accuracies,
        "final_train_loss": train_losses[-1],
        "final_val_loss"  : val_losses[-1],
        "final_val_acc"   : val_accuracies[-1],
        "perplexity"      : math.exp(val_losses[-1]),
        "train_time"      : train_time,
    }


In [11]:
def print_table(headers, rows):
    col_w = [max(len(str(h)), max(len(str(r[i])) for r in rows))
             for i, h in enumerate(headers)]
    fmt = "  ".join(f"{{:<{w}}}" for w in col_w)

    print(fmt.format(*headers))
    print("-" * (sum(col_w) + 2 * (len(headers) - 1)))
    
    for row in rows:
        print(fmt.format(*row))


# Question Parts

### Hyperparams

In [12]:
epochs = 10
learning_rate = 0.005
embed_size = 64
batch_size = 128
seed = "ROMEO:\n"

## Part 1

In [13]:
def part1():
    hidden_size = 128
    num_layers = 1
    seq_lengths = [20, 30]

    part_1 = {}

    for seq in seq_lengths:
        print('-'*70)
        print(f'Sequence Length: {seq}')
        print('-'*70)

        train_loader, test_loader = build_loaders(encoded_text, seq, batch_size)
        part_1[seq] = {}

        for label, modelClass in [("LSTM", modelLSTM), ("GRU", modelGRU)]:
            print('-'*70)
            print(f'\n  [{label}]')
            print('-'*70)

            model = modelClass(embed_size, hidden_size, num_layers).to(device)
            metrics = train_model(model, train_loader, test_loader, epochs, learning_rate)
            inf_time = measure_inference_time(model, test_loader)
            gen = gen_text(model, seed, seq, n_chars=120)

            part_1[seq][label] = {
                **metrics,
                "params": count_params(model),
                "size": model_size(model),
                "inf_s": inf_time,
                "generated": gen,
            }
    
    print("\n" + "=" * 90)
    print(f"Experiment 1 Results")
    print("=" * 90)

    for metric_name, key, fmt in [
        ("Final Train Loss",    "final_train_loss", ".4f"),
        ("Final Val Loss",      "final_val_loss",   ".4f"),
        ("Val Accuracy",        "final_val_acc",    ".4f"),
        ("Perplexity",          "perplexity",       ".2f"),
        ("Train Time (s)",      "train_time",       ".2f"),
        ("Inference Time (ms)", "inf_s",           ".3f"),
        ("Parameters",          "params",           ","),
        ("Model Size (MB)",     "size",          ".4f"),
    ]:
        rows = []
        for seq in seq_lengths:
            for label in ["LSTM", "GRU"]:
                v = part_1[seq][label][key]
                rows.append((f"seq={seq}", label, f"{v:{fmt}}"))
        print(f"\n{metric_name}")
        print_table(["Seq Len", "Model", "Value"], rows)
    
    print("\n--- Generated Text Samples ---")
    for seq in seq_lengths:
        for label in ["LSTM", "GRU"]:
            print(f"\n[seq={seq}  {label}]\n{part_1[seq][label]['generated']}")


## Part 2

In [14]:
def part2():
    seq_len = 20
    train_loader, test_loader = build_loaders(encoded_text, seq_len, batch_size)

    hp_config = [
        ("baseline", 128, 1, None),
        ("large_hidden", 256, 1, None),
        ("second_layer", 128, 2, None),
        ("fc_hidden", 128, 1, 256),
        ("hidden_layer_fc", 256, 2, 256)
    ]

    part_2 = {}

    for cfg, hidden, layers, fc_hidden in hp_config:
        part_2[cfg] = {}

        for label, modelClass in [("LSTM", modelLSTM), ("GRU", modelGRU)]:
            print(f'\n[{cfg}]  {label} hidden]{hidden} layers={layers} fc={fc_hidden}')

            model= modelClass(embed_size, hidden, layers, fc_hidden).to(device)
            metrics = train_model(model, train_loader, test_loader, epochs, learning_rate)
            inf_time = measure_inference_time(model, test_loader)
            gen = gen_text(model, seed, seq_len, n_chars=120)

            part_2[cfg][label] = {
                **metrics,
                "params"    : count_params(model),
                "size"   : model_size(model),
                "inf_s"    : inf_time,
                "hidden"    : hidden,
                "layers"    : layers,
                "fc_hidden" : fc_hidden,
                "generated" : gen
            }

    print("\n" + "=" * 90)
    print(f"Experiment 2 Results")
    print("=" * 90)

    for metric_name, key, fmt in [
        ("Final Train Loss",    "final_train_loss", ".4f"),
        ("Final Val Loss",      "final_val_loss",   ".4f"),
        ("Val Accuracy",        "final_val_acc",    ".4f"),
        ("Perplexity",          "perplexity",       ".2f"),
        ("Train Time (s)",      "train_time",       ".2f"),
        ("Inference Time (ms)", "inf_s",           ".3f"),
        ("Parameters",          "params",           ","),
        ("Model Size (MB)",     "size",          ".4f"),
    ]:
        rows = []
        for cfg, hidden, layers, fc_hidden in hp_config:
            for label in ["LSTM", "GRU"]:
                r = part_2[cfg][label]
                v = r[key]
                rows.append((cfg, label,
                            f"h={r['hidden']} l={r['layers']} fc={r['fc_hidden']}",
                            f"{v:{fmt}}"))
            print(f"\n{metric_name}")
            print_table(["Config", "Model", "Hyperparams", "Value"], rows)
 
    print("\n--- Generated Text Samples ---")
    for cfg, *_ in hp_config:
        for label in ["LSTM", "GRU"]:
            print(f"\n[{cfg}  {label}]\n{part_2[cfg][label]['generated']}")

## Part 3

In [15]:
def part3():
    seq_len = 50
    hidden_size = 128
    num_layers = 1
    train_loader, test_loader = build_loaders(encoded_text, seq_len, batch_size)

    print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

    part_3 = {}

    part_3 = {}
    for label, ModelClass in [("LSTM", modelLSTM), ("GRU", modelGRU)]:
        print(f"\n  [{label}]")
        model   = ModelClass(chars_len, embed_size, hidden_size, num_layers).to(device)
        metrics = train_model(model, train_loader, test_loader, epochs, learning_rate)
        inf_time   = measure_inference_time(model, test_loader)
        gen     = gen_text(model, (seed * 8)[:seq_len], seq_len, n_chars=120)
        part_3[label] = {
            **metrics,
            "params"   : count_params(model),
            "size"  : model_size(model),
            "inf_s"   : inf_time,
            "generated": gen,
        }

    print("\n" + "=" * 90)
    print(f"Experiment 3 Results")
    print("=" * 90)

    rows = []
    for label in ["LSTM", "GRU"]:
        r = part_3[label]
        rows.append((label,
                    f"{r['final_train_loss']:.4f}",
                    f"{r['final_val_loss']:.4f}",
                    f"{r['final_val_acc']:.4f}",
                    f"{r['perplexity']:.2f}",
                    f"{r['train_time']:.2f}s",
                    f"{r['inf_s']:.3f}ms",
                    f"{r['params']:,}",
                    f"{r['size']:.4f}"))
    print_table(["Model", "Train Loss", "Val Loss", "Val Acc", "Perplexity", "Train Time", "Inf Time", "Params", "Size"], rows)
    
    print("\n--- Generated Text Samples ---")
    for label in ["LSTM", "GRU"]:
        print(f"\n[seq=50  {label}]\n{part_3[label]['generated']}")


# Main Function

In [16]:
print("=" * 100)
print("EXPERIMENT 1: LSTM vs GRU | Seq lengths 20 and 30")
print("=" * 100)

part1()

print("=" * 100)
print("EXPERIMENT 2: HYPERPARAMETER SWEEP (seq_len=20)")
print("=" * 100)

part2()

print("=" * 100)
print("EXPERIMENT 3: SEQUENCE LENGTH 50")
print("=" * 100)

part3()

EXPERIMENT 1: LSTM vs GRU | Seq lengths 20 and 30
----------------------------------------------------------------------
Sequence Length: 20
----------------------------------------------------------------------
----------------------------------------------------------------------

  [LSTM]
----------------------------------------------------------------------
  Epoch   5 | train loss 1.5179 | val loss 1.5420 | val acc 0.5327
  Epoch  10 | train loss 1.5230 | val loss 1.5468 | val acc 0.5337
----------------------------------------------------------------------

  [GRU]
----------------------------------------------------------------------
  Epoch   5 | train loss 1.8402 | val loss 1.8776 | val acc 0.4503
  Epoch  10 | train loss 1.8579 | val loss 1.8682 | val acc 0.4541
----------------------------------------------------------------------
Sequence Length: 30
----------------------------------------------------------------------
-------------------------------------------------------